In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

In [0]:
#step-1
data=[("john,smith","Canada"),
      ("mary,jones","USA"),
      ("jose,lopez","Mexico")]
df=spark.createDataFrame(data,["Name","Country"])

display(df)

#step-2
df= df.withColumn("Name",split(col("Name"),","))
display(df)

#step-3

df = df.select(
    explode(col("Name")).alias("Name"),
    col("Country")
)

display(df)


In [0]:
df.explain(True)

In [0]:
schema = StructType([
    StructField("player", StringType(), True),
    StructField("runs", IntegerType(), True),
    StructField("50s/100s", StringType(), True)
])

#Create a DataFrame with the defined schema
data = [("Sachin-IND", 18694, "93/49"), ("Ricky-AUS", 11274, "66/31"),("Lara-WI", 10222, "45/21"),("Rahul-IND", 10355, "95/11"),("Jhonty-SA", 7051, "43/5"),("Hayden-AUS", 8722, "67/19")]
players_df = spark.createDataFrame(data, schema)

data1 = [("IND", "India"), ("AUS", "Australia"), ("WI", "WestIndies"), ("SA", "SouthAfrica")]
countries_df = spark.createDataFrame(data1,["SRT","country"])

display(players_df)
display(countries_df)
#step-1
 
players_df_new1 = players_df.withColumn("playerName",split(col("player"),"-").getItem(0)) \
    .withColumn("SRT",split(col("player"),"-").getItem(1)) \
    .withColumn("50s",split(col("50s/100s"),"/").getItem(0)) \
    .withColumn("100s",split(col("50s/100s"),"/").getItem(1)) \
    .withColumn("sums" , expr("int(`50s`)+int(`100s`)") ) \
    .filter(col("sums") > 90) \
    .select("playerName","runs","SRT","50s","100s","sums")                
display(players_df_new1)
 
#step-2
 
df_join = players_df_new1.join(countries_df,players_df_new1.SRT == countries_df.SRT,"inner")\
    .select("playerName","runs","country","sums")
display(df_join)
#step-2

In [0]:
df1_failfast = (
    spark.read
    .option("inferSchema", "true")
    .option("header", "true")
    .option("mode", "FAILFAST")
    .csv("/Volumes/workspace/default/my_files/employee.csv")
)
#display(df1_failfast)

#step-2
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

employee_schema = StructType([
    StructField("empid", IntegerType(), True),
    StructField("empname", StringType(), True),
    StructField("address", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])

df2_permissive = (
    spark.read
    .schema(employee_schema)
    .option("header", "true")
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .csv("/Volumes/workspace/default/my_files/employee.csv")
)

display(df2_permissive)
#step-3

df3_dropMalFormed = (
    spark.read
    .schema(employee_schema)
    .option("header", "true")
    .option("mode", "DROPMALFORMED")
    #.option("columnNameOfCorruptRecord", "_corrupt_record")
    .csv("/Volumes/workspace/default/my_files/employee.csv")
)

display(df3_dropMalFormed)

In [0]:
#Define the schema
schema = StructType([
    StructField("reqid", IntegerType(), True),
    StructField("pickup_location", StringType(), True)
])

#Create a DataFrame with the defined schema
data = [(48, "Airport"), (49, "Office"),(50, "Hospital"),(51, "Airport"),(52, "Hospital"),(53, "Shoppingmall"),(54, "Office"),(55, "Hospital"),(56, "Hospital")]
pickup_df = spark.createDataFrame(data, schema)
pickup_df.display()


#step-1:
pick_result =pickup_df.groupBy("pickup_location").count().orderBy(desc("count")).limit(3).select("pickup_location").display()

#2nd metod
pickup_result =pickup_df.groupBy("pickup_location").count()
pickup_window = pickup_result.withColumn("rrnum" , row_number().over(Window.orderBy(desc("count"))))
pickup_window.filter(pickup_window.rrnum <= 3).select("pickup_location").display()

In [0]:
#Sample data
data = [
    (100, 'IT', 100, '2024-05-12'),
    (200, 'IT', 100, '2024-06-12'),
    (100, 'FIN', 400, '2024-07-12'),
    (300, 'FIN', 500, '2024-07-12'),
    (300, 'FIN', 1543, '2024-07-12'),
    (300, 'FIN', 1500, '2024-07-12')
]

#Create DataFrame
columns = ["empid", "dept", "salary", "date"]
df = spark.createDataFrame(data, columns)
display(df)

#step-1
df_withcount = df.groupBy("empid").count().filter("count = 1") 
display(df_withcount)

#step-2
window_spec = Window.partitionBy('empid')

df_count = df.withColumn('count',count("*").over(window_spec))
df_count.filter("count == 1").drop("count").display()

#step-3

df_empid = df.groupBy("empid").agg(count("empid").alias("count"))
df_repeatedempid = df_empid.filter(col("count")>1).select("empid")
display(df_repeatedempid)
#using joins to get the elemnts

df_joins = df.join(df_repeatedempid, df.empid == df_repeatedempid.empid, "left_anti")
display(df_joins)



 

In [0]:
%sql
--Create table and insert data
delete from  emp_new;
CREATE TABLE IF NOT EXISTS emp_new (employee_id VARCHAR(50) );
INSERT INTO emp_new (employee_id) VALUES ('72657'),('1234'),('Tom'),('8792'),('Sam'),('19998'),('Philip')


In [0]:
%sql
select *,try_cast(employee_id as int) as employee_id from emp_new
where try_cast(employee_id as int) is not null;




In [0]:
from pyspark.sql.functions import col

# Load the source dataframe
df = spark.sql("select * from emp_new")

# Filter rows where employee_id can successfully be converted to an integer
df_filtered = df.filter(col('employee_id').cast('int').isNotNull())

# Display the resulting dataset
display(df_filtered)


In [0]:
#Sample data
from pyspark.sql.functions import col, upper, split,initcap
data = [("virat kohli",), ("p v sindhu",)]

#Create DataFrame
columns = ["name"]
df = spark.createDataFrame(data, columns)
display(df)

#first method
df_first = df.withColumn("name",initcap(col("name"))).display()

In [0]:
#Sample data for df1 and df2
data1 = [(1, 'Bob'), (2, 'Alice'), (3, 'Tom')]
data2 = [(1, 'Bob'), (3, 'Tom')]

#Create DataFrame
df1 = spark.createDataFrame(data1, ["id", "name"])
df2 = spark.createDataFrame(data2, ["id", "name"])

display(df1)
display(df2)
#method-1
df_join = df1.join(df2,["id","name"],"left_anti")
display(df_join)

#method-2
df1.subtract(df2).display()

In [0]:
data = [
    (10, "A", None),
    (None, "B", 30),
    (20, None, 40)
]

df = spark.createDataFrame(data, ["col1", "col2", "col3"])

from pyspark.sql import functions as F

df.select(
    F.sum(F.when(F.col("col1").isNull(), 1).otherwise(0)).alias("col1"),
    F.sum(F.when(F.col("col2").isNull(), 1).otherwise(0)).alias("col2"),
    F.sum(F.when(F.col("col3").isNull(), 1).otherwise(0)).alias("col3")
).show()

df.createOrReplaceTempView("tempDB")

In [0]:
data = [(1, None, 'ab'),
    (2, 10, None),
    (None, None, 'cd')]
columns = ['col1', 'col2', 'col3']
df = spark.createDataFrame(data, columns)
display(df)

df.createOrReplaceTempView("tempDB")

df.select([sum(col(c).isNull().cast('int')).alias(c) for c in df.columns]).display()




In [0]:
%sql

select * from tempDB;

 SELECT
  SUM(CASE WHEN col1 IS NULL THEN 1 ELSE 0 END) AS col1,
  SUM(CASE WHEN col2 IS NULL THEN 1 ELSE 0 END) AS col2,
  SUM(CASE WHEN col3 IS NULL THEN 1 ELSE 0 END) AS col3
FROM tempDB;



In [0]:
#Mentioning the dataframe details here
data = [(101, 'IT', 1000), (102, 'HR', 900)]
columns = ["empid", "dept", "salary"]
df = spark.createDataFrame(data, columns)
display(df)

prefix ='de_'

for old_col in df.columns:
    new_col = prefix + old_col
    df = df.withColumnRenamed(old_col,new_col)
display(df)    

In [0]:
%sql
--Create table syntax
drop table emps_tbl;

CREATE TABLE if not EXISTS emps_tbl (emp_name VARCHAR(50), dept_id INT, salary INT);

INSERT INTO emps_tbl VALUES ('Siva', 1, 30000), ('Ravi', 2, 40000), ('Prasad', 1, 50000), ('Sai', 2, 20000), ('Anna', 2, 10000);

WITH ranked_emps AS (
  SELECT
    *,
    ROW_NUMBER() OVER (PARTITION BY dept_id ORDER BY salary) AS min_sal,
    ROW_NUMBER() OVER (PARTITION BY dept_id ORDER BY salary DESC) AS max_sal
  FROM emps_tbl
)
SELECT
  dept_id,
  MAX(CASE WHEN min_sal = 1 THEN emp_name END) AS minsal_emp,
  MAX(CASE WHEN max_sal = 1 THEN emp_name END) AS maxsal_emp
FROM ranked_emps
GROUP BY dept_id;


In [0]:
%sql
--Create table syntax
drop table cards;
CREATE TABLE   if not exists cards (card_number BIGINT);
INSERT INTO cards VALUES (1234567812345678),(2345678923456789),(3456789034567890);

select * from cards;

--select * , replicate('*',12)  as new_cards from cards

select *,concat(REPEAT('*' , 12), right(card_number,4)) as new_cards from cards;




In [0]:
%sql
----------------------------------------------
drop table Employeetbl;

CREATE TABLE   if not exists Employeetbl (employee_id INT,ename VARCHAR(50),salary INT);
INSERT INTO Employeetbl VALUES (3, 'Bob', 60000),(4, 'Diana', 70000),
(5, 'Eve', 60000),
(6, 'Frank', 80000),
(7, 'Grace', 70000),
(8, 'Henry', 90000);

select * from employee;

select t1.ename
from employeetbl t1, Employeetbl t2
where t1.salary = t2.salary and t1.ename <> t2.ename;

In [0]:
%sql
--Create table syntax
CREATE TABLE if not exists transactions_1308 (transaction_id BIGINT, type VARCHAR(50), amount INT,transaction_date DATE);

-- Insert data into the table
INSERT INTO transactions_1308 VALUES (53151, 'deposit', 178, '2022-07-08'),
(29776, 'withdrawal', 25, '2022-07-08'),(16461, 'withdrawal', 45, '2022-07-08'),
(19153, 'deposit', 65, '2022-07-10'),(77134, 'deposit', 32, '2022-07-10');

with cte as (
	select * , ROW_NUMBER() OVER(ORDER BY transaction_date) AS rn ,
		(CASE
			when type = 'deposit' then amount else -amount
		END) AS trans
	from transactions_1308
	)
select transaction_id , type , amount , transaction_date , 
	SUM(trans) over(order by rn asc) AS balance_amount
from cte

In [0]:
%sql
--Create table syntax
CREATE TABLE if not exists Flights (cust_id INT, flight_id VARCHAR(10), origin VARCHAR(50), destination VARCHAR(50));

-- Insert data into the table
INSERT INTO Flights (cust_id, flight_id, origin, destination)
VALUES (1, 'SG1234', 'Delhi', 'Hyderabad'), 
(1, 'SG3476', 'Kochi', 'Mangalore'), 
(1, '69876', 'Hyderabad', 'Kochi'),
(2, '68749', 'Mumbai', 'Varanasi'),
(2, 'SG5723', 'Varanasi', 'Delhi');

--Method-1
with start_point_cte as (
select f.cust_id,f.origin
from flights f
where not exists (
     select 1 from flights x
	 where x.cust_id = f.cust_id and x.destination = f.origin   
	 )
)	 ,
end_point_cte as (
select cust_id,destination
from flights f
where not exists (
     select 1 from flights x
	 where x.cust_id = f.cust_id and x.origin = f.destination   
	 )
)
select s.cust_id,s.origin,e.destination
from start_point_cte s join  end_point_cte e
on s.cust_id = e.cust_id
order by s.cust_id;


--2nd method
with origin_flights_cte as (
select  f1.cust_id,f1.origin
from flights f1
left join flights f2
on f1.cust_id = f2.cust_id
and f1.origin = f2.destination
where f2.origin is null
),
destination_flights_cte as (

select  f1.cust_id,f1.destination 
from flights f1
left join flights f2
on f1.cust_id = f2.cust_id
and f1.destination = f2.origin
where f2.origin is null
)

select o.cust_id,o.origin,e.destination
from origin_flights_cte o join destination_flights_cte e 
on o.cust_id = e.cust_id;

In [0]:
result.explain(True)